<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 05 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">客户 Snapshot</div>
  <p class="doris-cover-lead">记录客户属性更新和硬删除，生成 SCD Type 2 历史以及当前客户维表。</p>
  <span class="doris-cover-note">Snapshot · check strategy · Hard Delete · SCD Type 2 · ref()</span>
</div>

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 dbt-for-apache-doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 5：客户 Snapshot

这个 Demo 分两轮展示 Snapshot：第一轮记录客户初始状态，第二轮在更新客户 1、删除客户 2 后写入新的 SCD Type 2 历史。

<div class="doris-flow">
  <div class="doris-flow-step"><strong>当前客户</strong>2 条源记录</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Staging View</strong><code>stg_customers</code></div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>第一次 Snapshot</strong>2 条有效历史</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>源数据变化</strong>更新客户 1，删除客户 2</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>SCD Type 2</strong>3 条历史、1 条当前客户</div>
</div>

### 2.1 准备并查看当前客户源表

源表使用 Unique Key 保存客户当前状态，初始包含 Alice 和 Bob 两条记录。

In [ ]:
snapshot_dir = runner.examples_root / "doris-demos/snapshot"
runner.show_file("Fixture SQL", snapshot_dir / "scripts/setup.sql")
runner.show_file("Source 声明", snapshot_dir / "models/sources.yml")
runner.run_sql_file("创建客户源表", snapshot_dir / "scripts/setup.sql")
runner.query("输入：客户当前状态", """
select customer_id, customer_number, customer_type, email
from dbt_demo_snapshot_source.CUSTOMERS
order by customer_id
""")

### 2.2 创建 staging View

`stg_customers` 通过 `source()` 读取当前客户表，为 Snapshot 提供稳定的上游节点。

In [ ]:
runner.show_file("客户 staging Model", snapshot_dir / "models/stg_customers.sql")
runner.run_dbt("创建 stg_customers View", snapshot_dir, "run", "--select", "stg_customers")
runner.query("中间结果：staging 客户", """
select customer_id, customer_type, email
from dbt_demo_snapshot.stg_customers
order by customer_id
""")

### 2.3 执行第一次 Snapshot

Snapshot 使用 `check` 策略监控邮箱、客户类型等字段，并开启 `invalidate_hard_deletes`。第一次执行会为两个客户各创建一个有效版本。

In [ ]:
runner.show_file("Snapshot 定义", snapshot_dir / "snapshots/customer_snapshot.sql")
runner.run_dbt("记录客户初始版本", snapshot_dir, "snapshot", "--select", "customer_snapshot", "--threads", "1")
runner.query("第一次 Snapshot：2 条有效历史", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")

### 2.4 从 Snapshot 生成当前客户维表

维表只保留 `dbt_valid_to is null` 的当前版本，并统计每个客户共有多少历史版本。第一次构建时两个客户都只有一个版本。

In [ ]:
runner.show_file("当前客户维表 Model", snapshot_dir / "models/dim_customer_current.sql")
runner.show_file("Data Test 定义", snapshot_dir / "models/snapshot.yml")
runner.run_dbt("创建当前客户维表", snapshot_dir, "run", "--select", "dim_customer_current")
runner.run_dbt("测试当前客户维表", snapshot_dir, "test", "--select", "dim_customer_current", "--threads", "1")
runner.query("第一次维表结果", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

### 2.5 修改和删除源客户

将客户 1 的邮箱和类型更新，并从当前源表删除客户 2。此时 Snapshot 历史尚未变化。

In [ ]:
customer_changes_sql = """
update dbt_demo_snapshot_source.CUSTOMERS
set email = 'alice.new@example.com', customer_type = 'INDIVIDUAL_PLUS'
where customer_id = 1;
delete from dbt_demo_snapshot_source.CUSTOMERS where customer_id = 2
"""
runner.show_sql("客户变更 SQL", customer_changes_sql)
runner.run_sql("更新 Alice 并删除 Bob", customer_changes_sql)
runner.query("变更后的当前源表", """
select customer_id, customer_type, email
from dbt_demo_snapshot_source.CUSTOMERS
order by customer_id
""")

### 2.6 执行第二次 Snapshot 并刷新维表

第二次 Snapshot 关闭 Alice 的旧版本并创建新版本，同时关闭已从源表删除的 Bob。随后刷新维表，只剩 Alice 的当前版本。

In [ ]:
runner.run_dbt("记录客户变更版本", snapshot_dir, "snapshot", "--select", "customer_snapshot", "--threads", "1")
runner.run_dbt("刷新当前客户维表", snapshot_dir, "run", "--select", "dim_customer_current")
runner.run_dbt("测试刷新后的客户维表", snapshot_dir, "test", "--select", "dim_customer_current", "--threads", "1")
runner.query("第二次 Snapshot：3 条历史", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")
runner.query("刷新后的当前客户维表", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

### 2.7 校验 SCD Type 2 历史

Verifier 检查 Alice 有一个关闭版本和一个当前版本，Bob 只有关闭版本，当前维表只包含 Alice。

In [ ]:
runner.run_script("校验 Snapshot Demo", snapshot_dir / "scripts/verify.sh")

## 完成

两轮 Snapshot、客户历史版本、硬删除处理和当前客户维表均已通过校验。